# Urban Transit ETL Pipeline  
## 01 - Data Collection and Initial Profiling

This notebook represents the first practical step in the **Urban Transit ETL Pipeline** project.

The objective of this notebook is to load the raw urban transit dataset, inspect its structure, and identify initial data quality issues before building the full ETL pipeline.

### Notebook Objectives

In this notebook, we will:

- Locate the raw CSV files inside the `data/raw/` directory.
- Read and combine the CSV files using Polars.
- Display the number of rows and columns.
- Inspect the first few records.
- Detect the dataset schema and data types.
- Check missing values.
- Check duplicate rows.
- Check duplicate trip IDs.
- Identify the date range of the dataset.
- Decide whether the dataset is suitable for the ETL pipeline.

## Step 1: Import Required Libraries

In this step, we import the required Python libraries.

We use:

- `pathlib.Path` to handle file paths in a clean and reliable way.
- `polars` for fast data loading, profiling, and processing.

Polars is useful for this project because it can process large datasets efficiently and supports high-performance dataframe operations.

In [2]:
from pathlib import Path
import polars as pl

## Step 2: Define the Raw Data Directory

In this step, we define the location of the raw dataset files.

The raw files are stored inside:

```text
data/raw/
```

The raw files should not be edited manually. All cleaning and transformation steps must be done through code to keep the pipeline reproducible.

In [3]:
RAW_DIR = Path("../data/raw")

csv_files = list(RAW_DIR.glob("*.csv"))

## Step 3: Read and Combine CSV Files

In this step, we search for all CSV files inside the raw data directory, read them using Polars, and combine them into a single dataframe.

The January 2024 Citi Bike dataset was extracted into two CSV files:

- `202401-citibike-tripdata_1.csv`
- `202401-citibike-tripdata_2.csv`

Both files have the same structure, so they can be combined vertically into one dataframe.

We also force station ID columns to be read as strings because IDs are identifiers, not numerical values used for calculations.

In [6]:
STATION_ID_SCHEMA = {
    "start_station_id": pl.Utf8,
    "end_station_id": pl.Utf8,
}

print("CSV files found:")
for file in csv_files:
    print(file.name)

df = pl.concat(
    [
        pl.read_csv(
            file,
            try_parse_dates=True,
            schema_overrides=STATION_ID_SCHEMA,
        )
        for file in csv_files
    ],
    how="vertical",
)

print("Rows:", df.height)
print("Columns:", df.width)

df.head()

CSV files found:
202401-citibike-tripdata_1.csv
202401-citibike-tripdata_2.csv
Rows: 1888085
Columns: 13


ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
str,str,datetime[μs],datetime[μs],str,str,str,str,f64,f64,f64,f64,str
"""8E865410DBDE0CA9""","""electric_bike""",2024-01-01 13:00:04.563,2024-01-01 13:04:04.652,"""3 St & 3 Ave""","""4028.03""","""Carroll St & Smith St""","""4225.14""",40.67507,-73.987752,40.680611,-73.994758,"""casual"""
"""0403D0B3FC9CA77D""","""electric_bike""",2024-01-08 19:36:43.520,2024-01-08 19:53:16.266,"""Franklin Ave & St Marks Ave""","""4107.05""","""Bedford Ave & Bergen St""","""4066.15""",40.675832,-73.956168,40.676368,-73.952918,"""casual"""
"""F6DE7BB42FF550BE""","""electric_bike""",2024-01-12 15:00:41.580,2024-01-12 15:36:29.622,"""W 67 St & Broadway""","""7116.04""","""Central Park W & W 103 St""","""7577.27""",40.774925,-73.982666,40.79559,-73.961884,"""casual"""
"""84A995BFD98030D4""","""classic_bike""",2024-01-12 16:52:19.025,2024-01-12 17:17:29.773,"""Central Park West & W 68 St""","""7079.06""","""E 5 St & Ave C""","""5545.04""",40.773407,-73.977825,40.722992,-73.979955,"""member"""
"""7BBEAD4F2B535813""","""electric_bike""",2024-01-05 19:50:19.202,2024-01-05 20:34:42.517,"""W 67 St & Broadway""","""7116.04""","""Ave A & E 14 St""","""5779.11""",40.774925,-73.982666,40.730311,-73.980472,"""member"""


## Step 4: Inspect Schema and Data Types

In this step, we inspect the detected schema of the dataset.

Schema inspection helps us understand whether each column has the expected data type before moving to schema validation and transformation.

This is important because incorrect data types can cause problems later during cleaning, transformation, Parquet conversion, or BigQuery loading.

In [7]:
schema_df = pl.DataFrame({
    "column_name": df.columns,
    "data_type": [str(dtype) for dtype in df.dtypes]
})

schema_df

column_name,data_type
str,str
"""ride_id""","""String"""
"""rideable_type""","""String"""
"""started_at""","""Datetime(time_unit='us', time_…"
"""ended_at""","""Datetime(time_unit='us', time_…"
"""start_station_name""","""String"""
…,…
"""start_lat""","""Float64"""
"""start_lng""","""Float64"""
"""end_lat""","""Float64"""


## Step 5: Check Missing Values

In this step, we calculate the number and percentage of missing values for each column.

Missing value detection is important because it helps us identify data quality issues that should be handled later in the cleaning stage.

At this point, we are only profiling the data. We are not cleaning or changing the raw dataset yet.

In [9]:
missing_summary = []

for col in df.columns:
    null_count = df[col].null_count()
    null_percentage = (null_count / df.height) * 100

    missing_summary.append({
        "column_name": col,
        "missing_count": null_count,
        "missing_percentage": round(null_percentage, 2)
    })

pl.DataFrame(missing_summary)

column_name,missing_count,missing_percentage
str,i64,f64
"""ride_id""",0,0.0
"""rideable_type""",0,0.0
"""started_at""",0,0.0
"""ended_at""",0,0.0
"""start_station_name""",1160,0.06
…,…,…
"""start_lat""",1160,0.06
"""start_lng""",1160,0.06
"""end_lat""",5486,0.29


## Step 6: Check Duplicate Records

In this step, we check for duplicated full rows.

Duplicate records can cause incorrect analysis results, especially when calculating total trips, station activity, or rider distribution.

In [10]:
duplicate_count = df.height - df.unique().height

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


## Step 7: Check Duplicate Trip IDs

In this step, we check whether the `ride_id` column contains duplicate values.

The `ride_id` column is expected to uniquely identify each trip. If no duplicate `ride_id` values are found, it can be treated as a candidate primary key.

In [11]:
ride_id_duplicates = df.height - df.select("ride_id").unique().height

print("Duplicate ride_id values:", ride_id_duplicates)

Duplicate ride_id values: 0


## Step 8: Check Dataset Date Range

In this step, we check the minimum and maximum values for the `started_at` and `ended_at` columns.

This helps us confirm that the dataset mostly covers January 2024 and allows us to identify any unusual dates.

In [12]:
df.select([
    pl.col("started_at").min().alias("min_started_at"),
    pl.col("started_at").max().alias("max_started_at"),
    pl.col("ended_at").min().alias("min_ended_at"),
    pl.col("ended_at").max().alias("max_ended_at")
])

min_started_at,max_started_at,min_ended_at,max_ended_at
datetime[μs],datetime[μs],datetime[μs],datetime[μs]
2023-12-31 02:36:55.648,2024-01-31 23:58:30.270,2024-01-01 00:00:08.272,2024-01-31 23:59:56.370


## Step 9: Display Full Missing Values Report

In this step, we display the full missing values report without truncating rows or columns.

This report will be used later in the documentation and cleaning stage.

In [13]:
missing_df = pl.DataFrame(missing_summary)

with pl.Config(tbl_rows=50, tbl_cols=10):
    print(missing_df)

shape: (13, 3)
┌────────────────────┬───────────────┬────────────────────┐
│ column_name        ┆ missing_count ┆ missing_percentage │
│ ---                ┆ ---           ┆ ---                │
│ str                ┆ i64           ┆ f64                │
╞════════════════════╪═══════════════╪════════════════════╡
│ ride_id            ┆ 0             ┆ 0.0                │
│ rideable_type      ┆ 0             ┆ 0.0                │
│ started_at         ┆ 0             ┆ 0.0                │
│ ended_at           ┆ 0             ┆ 0.0                │
│ start_station_name ┆ 1160          ┆ 0.06               │
│ start_station_id   ┆ 1160          ┆ 0.06               │
│ end_station_name   ┆ 5505          ┆ 0.29               │
│ end_station_id     ┆ 5505          ┆ 0.29               │
│ start_lat          ┆ 1160          ┆ 0.06               │
│ start_lng          ┆ 1160          ┆ 0.06               │
│ end_lat            ┆ 5486          ┆ 0.29               │
│ end_lng            ┆ 54

## Initial Profiling Summary

The dataset was successfully loaded and profiled using Polars.

### Summary of Findings

| Item | Result |
|---|---:|
| CSV files loaded | 2 |
| Rows | 1,888,085 |
| Columns | 13 |
| Duplicate rows | 0 |
| Duplicate `ride_id` values | 0 |

### Missing Values Found

| Column | Missing Count | Missing Percentage |
|---|---:|---:|
| `start_station_name` | 1,160 | 0.06% |
| `start_station_id` | 1,160 | 0.06% |
| `end_station_name` | 5,505 | 0.29% |
| `end_station_id` | 5,505 | 0.29% |
| `start_lat` | 1,160 | 0.06% |
| `start_lng` | 1,160 | 0.06% |
| `end_lat` | 5,486 | 0.29% |
| `end_lng` | 5,486 | 0.29% |

### Date Range

| Field | Value |
|---|---|
| Minimum `started_at` | 2023-12-31 02:36:55.648 |
| Maximum `started_at` | 2024-01-31 23:58:30.270 |
| Minimum `ended_at` | 2024-01-01 00:00:08.272 |
| Maximum `ended_at` | 2024-01-31 23:59:56.370 |

One trip start timestamp appears on `2023-12-31`, which is acceptable because some trips may start before midnight and end in January 2024.

## Data Collection Decision

Based on the initial profiling results, the **NYC Citi Bike January 2024** dataset is suitable for the **Urban Transit ETL Pipeline** project.

The dataset is suitable because:

- It is a real urban transit dataset.
- It has a clear and consistent structure.
- It contains more than 1.8 million records, which is useful for demonstrating batch processing.
- It contains timestamps, station information, coordinates, and rider types.
- It contains some missing values, which makes it useful for demonstrating data cleaning.
- It has no duplicate full rows or duplicate trip IDs.
- It can support meaningful BigQuery SQL analysis.

Therefore, this dataset will be used as the raw input for the next ETL stages.

## Next Step

The next step is to build the **Data Ingestion module**.

Instead of keeping the logic only inside this notebook, the code will be moved into reusable Python modules inside the `src/` directory.

The next stage will focus on:

- Reading raw CSV files from a configured path.
- Handling file errors.
- Validating file existence.
- Loading the dataset in a reusable way.
- Preparing the dataframe for schema validation.